# Morfeus en Colab

Generador automático de videos virales cortos (9:16) tipo TikTok/Reels/Shorts. Todo gratis, todo en la nube.

**Pasos:**
1. Subir el código (clonar repo o subir como ZIP a `/content/`).
2. Instalar dependencias.
3. Configurar tu API key de Groq (gratis, https://console.groq.com).
4. (Opcional) Montar tu Drive para guardar outputs y assets.
5. Generar un video desde un producto.

**Tip:** activa GPU en `Runtime → Change runtime type → T4 GPU` sólo si vas a usar `--lipsync` (es el único paso que la necesita).

## 1. Subir el código a Colab

Si ya subiste tu repo a GitHub, descomenta la primera línea. Si no, sube el ZIP del proyecto desde la barra lateral (📁 → ⬆️) y descomprime.

In [ ]:
# Opción A — desde GitHub (cambia la URL):
# !git clone https://github.com/TU_USUARIO/proyecto-morfeus.git /content/morfeus_repo

# Opción B — desde un ZIP subido a /content/:
# !unzip -q /content/proyecto-morfeus.zip -d /content/morfeus_repo

# Si ya subiste el repo descomprimido a /content/morfeus_repo, sólo:
%cd /content/morfeus_repo
!ls

## 2. Instalar dependencias

Colab ya trae ffmpeg. Instalamos Morfeus + los extras de LLM y trends.

In [ ]:
!pip install -q -e ".[all]"
!morfeus --version

## 3. API keys

Necesitas **una** de las siguientes (las dos son gratis):
- **Groq** (recomendado): https://console.groq.com → Create API Key
- **Gemini** (fallback): https://aistudio.google.com → Get API Key

Pegá tu key en la celda siguiente. **No la commitees al repo.**

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Groq API key (deja vacío si usarás Gemini): ").strip()
if not os.environ.get("GROQ_API_KEY") and not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Gemini API key: ").strip()

## 4. (Opcional) Montar Google Drive

Si querés que los outputs y los assets de plantillas vivan en tu Drive (para reutilizar entre sesiones de Colab).

In [ ]:
from google.colab import drive  # noqa
drive.mount('/content/drive')

import os
OUTPUTS_DIR = '/content/drive/MyDrive/Morfeus/outputs'
TEMPLATES_DIR = '/content/drive/MyDrive/Morfeus/templates'
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(TEMPLATES_DIR, exist_ok=True)
os.environ['MORFEUS_TEMPLATES_DIR'] = TEMPLATES_DIR
print('OUTPUTS_DIR:', OUTPUTS_DIR)
print('MORFEUS_TEMPLATES_DIR:', os.environ['MORFEUS_TEMPLATES_DIR'])

## 5. Verificación rápida — demo estático sin LLM

Genera un MP4 corto a partir del `script_demo.json` empaquetado, sin llamar al LLM ni al trend scout. Útil para validar que el pipeline de TTS + composición funciona.

In [ ]:
!morfeus generate-demo --out /content/morfeus_demo.mp4 --static
from IPython.display import Video
Video('/content/morfeus_demo.mp4', embed=True, width=360)

## 6. Listar trends actuales

El Trend Scout junta Google Trends + (best-effort) TikTok Creative Center y los cachea 24h.

In [ ]:
!morfeus trends list --region MX --limit 10

## 7. Generar un video real desde un producto

Pasa `--product` con la descripción de tu negocio/producto. El sistema:
1. Detecta el trend más relevante.
2. Le pide al LLM (Groq) un guion siguiendo el formato de la plantilla.
3. Sintetiza voces con edge-tts.
4. Compone el video animado con bobbing + karaoke.

In [ ]:
PRODUCTO = "Barbería Don Carlos en Tegucigalpa, cortes premium con mascarilla y café gratis"
OUT = '/content/drive/MyDrive/Morfeus/outputs/barberia_demo.mp4' if os.path.exists('/content/drive/MyDrive') else '/content/barberia_demo.mp4'

!morfeus generate \
    --product "{PRODUCTO}" \
    --template socrates_skeleton \
    --region MX \
    --animated \
    --save-script /content/last_script.json \
    --out "{OUT}"

from IPython.display import Video
Video(OUT, embed=True, width=360)

## 8. (Opcional) Lip-sync con SadTalker

Lento (3-5 min por turno) y necesita GPU. Sólo recomendado para una versión final cuando el guion ya te gusta.

In [ ]:
# Sólo correr si quieres lip-sync — y asegurate de tener Runtime → GPU.
# !git clone https://github.com/OpenTalker/SadTalker.git /content/SadTalker
# %cd /content/SadTalker
# !pip install -q -r requirements.txt
# !bash scripts/download_models.sh
# %cd /content/morfeus_repo
# import os
# os.environ['MORFEUS_SADTALKER_DIR'] = '/content/SadTalker'

In [ ]:
# Generar con lip-sync activado:
# !morfeus generate \
#     --product "{PRODUCTO}" \
#     --template socrates_skeleton \
#     --animated --lipsync \
#     --out /content/barberia_lipsync.mp4

## Tips

- **Cambiar plantilla:** `--template abuelita_nieto` (otro formato incluido). O crea la tuya con `morfeus templates new mi_trend --in $MORFEUS_TEMPLATES_DIR`.
- **Forzar un tono específico:** `--trend "sátira política, ritmo lento, tono solemne"` (anula el scout).
- **Guardar el guion antes de renderizar:** `--save-script script.json`. Después podés editarlo a mano y re-renderizar con `--script script.json`.
- **Imágenes de personajes:** colócalas en `morfeus/templates/<nombre>/assets/socrates.png` (o el equivalente). Sin ellas se usan placeholders con el nombre.